In [ ]:
import glob
import os
import pandas as pd
from catboost import CatBoostClassifier
from pymatgen.core import Composition, Element

In [ ]:
aflow = pd.read_csv('./data/aflow_features.csv')

In [ ]:
pod_features = [
    'pearson',
    'space',
    'volume',
    'density',
    
    'band_gap',
    'efermi',
    'formation_energy_per_atom',
    
    'MagpieData mean NsValence',
    'MagpieData mean NpValence',
    'MagpieData mean NdValence',
    'MagpieData mean NfValence',
    
    'MagpieData mean NUnfilled',
    
    'MagpieData mean CovalentRadius',
    'MagpieData mean AtomicWeight',
    
    'MagpieData mean Electronegativity',
    'MagpieData mean GSmagmom',
    
    'MagpieData mean Row',
    'MagpieData mean Column'
]

In [ ]:
model_dir = "./model"
model_paths = sorted(glob.glob(os.path.join(model_dir, "*.cbm")))

top_models = []
for path in model_paths:
    model = CatBoostClassifier()
    model.load_model(path)
    top_models.append(model)

In [ ]:
for i, model in enumerate(top_models, start=1):
    col_name = f"model_{i}_score"

    aflow[col_name] = model.predict_proba(aflow[pod_features])[:, 1]

model_score_cols = [f"model_{i}_score" for i in range(1, len(top_models) + 1)]

aflow["avg_score"] = aflow[model_score_cols].mean(axis=1)
aflow["std_score"] = aflow[model_score_cols].std(axis=1)

aflow = aflow.sort_values(by="avg_score", ascending=False).reset_index(drop=True)
aflow.to_csv('./data/predict_aflow.csv', index=False)

In [ ]:
len(aflow)

In [ ]:
aflow_v2 = aflow[aflow['avg_score'] > 0.8]

In [ ]:
len(aflow_v2)

In [ ]:
score_cols = [col for col in aflow.columns if col.startswith("model_") and col.endswith("_score")]
mask = (aflow[score_cols] > 0.50).all(axis=1)
aflow_v3 = aflow_v2[mask]

In [ ]:
len(aflow_v3)

In [ ]:
aflow_v3['formula'].head(50).unique()

In [ ]:
def is_rare_earth(el: Element) -> bool:
    return el.is_lanthanoid or el.symbol in ["Sc", "Y"]

def has_one_tm_and_one_re(formula: str) -> bool:
    elems = Composition(formula).elements
    re_count = sum(is_rare_earth(el) for el in elems)
    tm_count = sum(
        el.is_transition_metal and not el.is_lanthanoid and el.symbol not in ["Sc", "Y"]
        for el in elems
    )
    return (re_count >= 1) and (tm_count >= 1)

mask = aflow_v3['formula'].apply(has_one_tm_and_one_re)
aflow_v4 = aflow_v3[mask]

In [ ]:
len(aflow_v4)